<a href="https://colab.research.google.com/github/raymondtyaba/case-manager-dashboard/blob/main/project2_eda_describe_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 Phase 1 EDA: Describe the Data
Upload the CDC PLACES `.xlsx` file when prompted, then run all cells in order.

In [ ]:
!pip install -q pandas openpyxl

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from google.colab import files
print("Libraries are ready")

Libraries are ready


In [ ]:
uploaded = files.upload()
print("Upload completed")

Saving CDC_PLACES_County_Data_2025.xlsx to CDC_PLACES_County_Data_2025.xlsx
Upload completed


In [ ]:
excel_files = [p for p in Path('/content').glob('*.xlsx') if not p.name.startswith('~$')]
if not excel_files:
    raise FileNotFoundError("No .xlsx file found. Upload the CDC PLACES Excel file and rerun this cell.")
preferred = [p for p in excel_files if 'cdc' in p.name.lower() or 'place' in p.name.lower()]
file_path = preferred[0] if preferred else excel_files[0]
book = pd.ExcelFile(file_path, engine='openpyxl')
df = pd.read_excel(file_path, sheet_name=book.sheet_names[0], engine='openpyxl')
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
print(f"Loaded: {file_path.name}")
print(f"Shape: {df.shape}")
display(df.head())

Loaded: CDC_PLACES_County_Data_2025.xlsx
Shape: (229298, 22)


,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
0,2023,AR,Arkansas,Drew,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,29.9,...,26.6,33.3,16945,13230,5043,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-91.7196579038053 33.5894113647675)
1,2023,AR,Arkansas,Fulton,BRFSS,Health Outcomes,Current asthma among adults,%,Crude prevalence,10.6,...,9.2,11.9,12421,9795,5049,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,POINT (-91.817888079321 36.3816206347765)
2,2023,AR,Arkansas,Howard,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,31.2,...,27.7,34.8,12533,9311,5061,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-93.9938053308515 34.0889044688856)
3,2023,AR,Arkansas,Miller,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,4.7,...,4.2,5.3,42415,32396,5091,HLTHOUT,STROKE,CrdPrv,Stroke,POINT (-93.8924281761086 33.3123156126392)
4,2023,AR,Arkansas,Ouachita,BRFSS,Disability,Any disability among adults,%,Crude prevalence,42.8,...,37.9,47.8,21793,16948,5103,DISABLT,DISABILITY,CrdPrv,Any Disability,POINT (-92.882040791321 33.593253927504)


## Step 1: Descriptive statistics

In [ ]:
print("Descriptive statistics for numeric columns")
display(df.select_dtypes(include='number').describe().T)

Descriptive statistics for numeric columns


,count,mean,std,min,25%,50%,75%,max
Year,229298.0,2022.862842,3.440146e-01,2022.0,2023.0,2023.0,2023.0,2023.0
Data_Value,229232.0,25.988058,2.207962e+01,1.7,9.8,17.6,34.4,93.3
Low_Confidence_Limit,229232.0,23.056486,2.095688e+01,1.5,8.3,14.8,29.4,92.2
High_Confidence_Limit,229232.0,29.029283,2.314743e+01,1.9,11.5,20.6,39.5,94.2
TotalPopulation,229298.0,223455.162138,6.261820e+06,43.0,10581.0,25915.0,69460.0,334914895.0
TotalPop18plus,229298.0,174826.488194,4.900162e+06,30.0,8240.0,20528.0,54087.0,262083034.0
LocationID,229298.0,30051.232353,1.526544e+04,59.0,18079.0,29151.0,45047.0,56045.0


## Locate and prepare the proposed research variables

In [ ]:
def find_col(exact, terms=None):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if terms:
        for c in df.columns:
            label = str(c).strip().lower()
            if all(t.lower() in label for t in terms):
                return c
    return None

measure_col = find_col(['measure'])
value_col = find_col(['data_value','data value'], ['data','value'])
type_col = find_col(['data_value_type','data value type'], ['value','type'])
year_col = find_col(['year'])
if measure_col is None or value_col is None:
    raise KeyError("Required Measure or Data_Value column not found. Confirm the correct worksheet was loaded.")
measure_text = df[measure_col].astype('string')
numeric_value = pd.to_numeric(df[value_col], errors='coerce')
print('Measure column:', measure_col)
print('Value column:', value_col)
print('Value type column:', type_col)

Measure column: Measure
Value column: Data_Value
Value type column: Data_Value_Type


In [ ]:
diabetes_mask = measure_text.str.contains('diabetes', case=False, na=False)
food_mask = measure_text.str.contains('food insecurity', case=False, na=False)
if type_col is not None:
    age_mask = df[type_col].astype('string').str.contains('age-adjusted', case=False, na=False)
    if (diabetes_mask & age_mask).any(): diabetes_mask &= age_mask
    if (food_mask & age_mask).any(): food_mask &= age_mask

def latest(mask):
    if year_col is None or not mask.any(): return mask
    years = pd.to_numeric(df.loc[mask, year_col], errors='coerce')
    return mask & (pd.to_numeric(df[year_col], errors='coerce') == years.max()) if years.notna().any() else mask

diabetes_values = numeric_value[latest(diabetes_mask)].dropna()
food_values = numeric_value[latest(food_mask)].dropna()
if diabetes_values.empty: raise ValueError("No numeric diabetes records found.")
if food_values.empty: raise ValueError("No numeric food-insecurity records found.")
research_desc = pd.DataFrame({'Diabetes prevalence':diabetes_values.describe(), 'Food insecurity prevalence':food_values.describe()})
display(research_desc)

,Diabetes prevalence,Food insecurity prevalence
count,2957.000000,2300.000000
mean,11.167907,17.297652
std,2.329725,6.094196
min,6.000000,5.900000
25%,9.500000,12.900000
50%,10.700000,15.800000
75%,12.500000,20.025000
max,23.800000,46.800000


## Step 2: Interpret descriptive output

In [ ]:
def explain(name, s):
    d=s.describe(); mean=d['mean']; median=d['50%']
    shape='approximately balanced'
    if median != 0 and mean > 1.05*median: shape='possibly right-skewed'
    elif median != 0 and mean < 0.95*median: shape='possibly left-skewed'
    print(f"{name}: n={int(d['count']):,}, mean={mean:.2f}, median={median:.2f}, std={d['std']:.2f}, min={d['min']:.2f}, max={d['max']:.2f}. The distribution is {shape}.")
explain('Diagnosed diabetes prevalence', diabetes_values)
explain('Food insecurity prevalence', food_values)

Diagnosed diabetes prevalence: n=2,957, mean=11.17, median=10.70, std=2.33, min=6.00, max=23.80. The distribution is approximately balanced.
Food insecurity prevalence: n=2,300, mean=17.30, median=15.80, std=6.09, min=5.90, max=46.80. The distribution is possibly right-skewed.


## Step 3: Proportional distribution for a categorical variable

In [ ]:
category_col = type_col if type_col is not None else measure_col
proportions = df[category_col].value_counts(normalize=True, dropna=False)
display(proportions.rename('proportion').to_frame())
print('Proportions sum to:', round(float(proportions.sum()),6))

,proportion
Data_Value_Type,
Crude prevalence,0.5
Age-adjusted prevalence,0.5


Proportions sum to: 1.0


## Step 4: Missing-data check

In [ ]:
missing = pd.DataFrame({'missing_count':df.isnull().sum(), 'missing_percent':(df.isnull().mean()*100).round(2)}).sort_values('missing_count',ascending=False)
display(missing)
most_missing = missing.index[0]
print(f"Most missing: {most_missing}, {int(missing.iloc[0]['missing_count']):,} values ({missing.iloc[0]['missing_percent']:.2f}%).")

,missing_count,missing_percent
Data_Value_Footnote_Symbol,229218,99.97
Data_Value_Footnote,229218,99.97
LocationName,80,0.03
Geolocation,80,0.03
Data_Value,66,0.03
High_Confidence_Limit,66,0.03
Low_Confidence_Limit,66,0.03
Category,0,0.00
StateDesc,0,0.00
StateAbbr,0,0.00


Most missing: Data_Value_Footnote_Symbol, 229,218 values (99.97%).


## Checkpoint: Three-variable comparison

In [ ]:
series_map={'Diagnosed diabetes prevalence':diabetes_values,'Food insecurity prevalence':food_values,'All numeric Data_Value records':numeric_value.dropna()}
rows=[]
for name,s in series_map.items():
    mean=float(s.mean()); std=float(s.std()); cv=abs(std/mean) if mean else float('nan')
    rows.append({'Variable':name,'Count':int(s.count()),'Mean':mean,'Standard deviation':std,'Coefficient of variation':cv})
comparison=pd.DataFrame(rows)
display(comparison)
most_variable=comparison.loc[comparison['Coefficient of variation'].idxmax(),'Variable']
print('Greatest relative variation:',most_variable)
print('Most missing full-dataset column:',most_missing)

,Variable,Count,Mean,Standard deviation,Coefficient of variation
0,Diagnosed diabetes prevalence,2957,11.167907,2.329725,0.208609
1,Food insecurity prevalence,2300,17.297652,6.094196,0.352313
2,All numeric Data_Value records,229232,25.988058,22.079617,0.849606


Greatest relative variation: All numeric Data_Value records
Most missing full-dataset column: Data_Value_Footnote_Symbol


## Step 5: Sanity check

In [ ]:
def flags(name,s):
    d=s.describe(); iqr=d['75%']-d['25%']; lower=d['25%']-1.5*iqr; upper=d['75%']+1.5*iqr
    outliers=int(((s<lower)|(s>upper)).sum()); findings=[]
    if d['50%'] and abs(d['mean']-d['50%'])/abs(d['50%'])>0.20: findings.append('mean differs from median by more than 20%')
    if outliers: findings.append(f'{outliers:,} values fall outside 1.5-IQR bounds')
    if d['min']<0 or d['max']>100: findings.append('values occur outside the expected 0-100 percentage range')
    print(name+': '+('; '.join(findings) if findings else 'no obvious issue detected by these checks'))
flags('Diagnosed diabetes prevalence',diabetes_values)
flags('Food insecurity prevalence',food_values)

Diagnosed diabetes prevalence: 73 values fall outside 1.5-IQR bounds
Food insecurity prevalence: 86 values fall outside 1.5-IQR bounds


## Final checkpoint summary

In [ ]:
print(f"Phase 1 EDA confirmed usable numeric variation in diabetes and food-insecurity prevalence across counties. {most_variable} had the greatest relative variation among the three inspected series. The full-dataset column with the most true missing data was {most_missing}, which should be reviewed during preparation. These descriptive results identify possible research variables but do not demonstrate correlation or causation.")

Phase 1 EDA confirmed usable numeric variation in diabetes and food-insecurity prevalence across counties. All numeric Data_Value records had the greatest relative variation among the three inspected series. The full-dataset column with the most true missing data was Data_Value_Footnote_Symbol, which should be reviewed during preparation. These descriptive results identify possible research variables but do not demonstrate correlation or causation.
